# 09 - HDFS

Paste each Bash block into an Ubuntu terminal in order. Keep one terminal for this notebook. Complete monitoring and the shared-image/workbench notebook. This uses one NameNode and one DataNode. The NameNode is formatted only when its data directory is empty.

## Open your workspace

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Services: namenode:9000 (RPC), namenode:9870 (web/metrics), datanode:9864 (web/metrics). PVCs: data-namenode-0 and data-datanode-0, each 5 GiB, mounted at /data. Namespace data and block data must be retained together. Native HDFS metrics are scraped by Prometheus; Alloy collects daemon logs.

## Apply telemetry settings

```bash
kubectl apply -f monitoring/telemetry-config.yaml
```



## Apply HDFS

```bash
kubectl apply -f hadoop/hdfs.yaml
```



## Wait for the NameNode

```bash
kubectl rollout status statefulset/namenode --timeout=600s
```



## Wait for the DataNode

```bash
kubectl rollout status statefulset/datanode --timeout=600s
```



## Wait for safe mode to finish

```bash
kubectl exec namenode-0 -- hdfs dfsadmin -safemode wait
```

Do not force safe mode off; the DataNode must register normally.

## Create working directories

```bash
kubectl exec namenode-0 -- hdfs dfs -mkdir -p /user/team/test-hdfs /tmp/hive /user/hive/warehouse /user/hive/external /spark-history
kubectl exec namenode-0 -- hdfs dfs -chmod 1777 /tmp /tmp/hive
```



## Write a small file

```bash
printf 'workspace hdfs example\n' > .state/test-hdfs.txt
kubectl exec -i namenode-0 -- hdfs dfs -put -f - /user/team/test-hdfs/input.txt < .state/test-hdfs.txt
```



## Read and compare the file

```bash
kubectl exec namenode-0 -- hdfs dfs -cat /user/team/test-hdfs/input.txt > .state/read-hdfs.txt
cmp .state/test-hdfs.txt .state/read-hdfs.txt
cat .state/read-hdfs.txt
```



## Inspect HDFS

```bash
kubectl exec namenode-0 -- hdfs dfsadmin -report
kubectl get pvc data-namenode-0 data-datanode-0
```

Expect one live DataNode.

## Check collected logs and metrics

```bash
python3 validations/check-observability.py --components namenode,datanode --jobs hdfs-namenode,hdfs-datanode
```

The helper makes temporary loopback API forwards, waits for actual log/metric samples and closes its forwards. It fails if collection is missing. Allow several scrape intervals after startup.

## Open the HDFS browser

Run this in a second terminal and keep it open. Continue the setup in your original terminal.

```bash
kubectl port-forward --address=127.0.0.1 service/namenode 19870:9870
```

Keep this second terminal open; use http://127.0.0.1:19870 in the Guacamole browser.

## Stop HDFS

```bash
bash stop-components.sh hdfs
```

Stop dependent YARN/Hive jobs before this command. PVCs remain.

## Resume HDFS

```bash
bash start-components.sh hdfs
```



## Check the retained file

```bash
kubectl exec namenode-0 -- hdfs dfs -cat /user/team/test-hdfs/input.txt > .state/read-hdfs.txt
cmp .state/test-hdfs.txt .state/read-hdfs.txt
```

